# CosyVoice 2 · 개인 코랩

**런타임 → 런타임 유형 변경 → T4 GPU**를 선택하고, **1번 왼쪽 ▶**를 누르세요.
`준비 완료`가 나오면 사이트 연결은 **4번**, 코랩 안에서 직접 만들기는 **2·3번**을 사용하세요.

설치는 처음에 시간이 걸립니다. 실패하면 멈추고 실제 오류를 셀 아래에 표시합니다.
같은 런타임에서 다시 실행하면 기존 설치와 모델 다운로드를 재사용합니다.
참조 음성은 배경음 없이 한 사람만 말하는 **3~30초 WAV/MP3(10MB 이하)**와 정확한 실제 대사를 준비하세요.


In [ ]:
#@title 1. CosyVoice 2 설치 및 실행 · v2.8.1 · 서버 실행 환경 수정본
from pathlib import Path
from collections import deque
import subprocess
import sys
server_source = (
    '# -*- coding: utf-8 -*-\n"""Self-contained CosyVoice 2 Colab runner. No edits to upstream model c'
    'ode.\n\n--setup: install an isolated Python 3.10 environment and start the local API.\n--tunnel: op'
    'tionally expose the ready API to AI Voice Studio.\n--serve: internal worker, launched with the is'
    'olated Python interpreter.\n"""\nimport argparse\nimport hashlib\nimport io\nimport json\nimport loggi'
    'ng\nimport os\nfrom pathlib import Path\nimport re\nimport secrets\nimport shutil\nimport socket\nimpor'
    't subprocess\nimport sys\nimport tempfile\nimport threading\nimport time\nimport traceback\nimport url'
    "lib.request\n\nSOURCE_REVISION = '074ca6dc9e80a2f424f1f74b48bdd7d3fea531cc'\nCOSY_MODEL_REVISION = "
    "'eec1ae6c79877dbd9379285cf8789c9e0879293d'\n\n# Retain the existing installation/cache path so an "
    "interrupted install can resume.\nENGINE = 'cosyvoice'\nLABEL = 'CosyVoice 2'\nROOT = Path('/content"
    "/ai_voice_dual_v1/cosyvoice')\nSOURCE = ROOT / 'CosyVoice'\nPYTHON = ROOT / 'venv/bin/python'\nMODE"
    "L = SOURCE / 'pretrained_models/CosyVoice2-0.5B'\nSTATE = ROOT / 'state.json'\nSERVICE = 'ai-voice"
    "-studio-cosyvoice'\nMODEL_REVISION = COSY_MODEL_REVISION\n\n\ndef run(args, label):\n    print('\\n▶ '"
    " + label, flush=True)\n    with (ROOT / 'setup.log').open('a', encoding='utf-8') as log:\n        "
    "log.write('\\n▶ ' + label + '\\n')\n        log.flush()\n        proc = subprocess.Popen([str(a) for"
    ' a in args], stdout=subprocess.PIPE,\n                                stderr=subprocess.STDOUT, t'
    'ext=True, bufsize=1)\n        try:\n            for line in proc.stdout:\n                print(lin'
    "e, end='', flush=True)\n                log.write(line)\n                log.flush()\n            c"
    'ode = proc.wait()\n        except BaseException:\n            proc.terminate()\n            raise\n '
    "   if code:\n        raise RuntimeError(f'{label} 실패 (종료 코드 {code}). 바로 위 오류를 확인해주세요.')\n\n\ndef rea"
    "d_state():\n    try:\n        return json.loads(STATE.read_text(encoding='utf-8'))\n    except (OSE"
    'rror, ValueError):\n        return {}\n\n\ndef health(base):\n    with urllib.request.urlopen(base + '
    "'/health', timeout=5) as response:\n        data = json.load(response)\n    return data.get('servi"
    "ce') == SERVICE and data.get('ready') is True\n\n\ndef setup():\n    if not Path('/content').is_dir("
    "):\n        raise RuntimeError('이 파일은 Google Colab에서 실행해주세요.')\n    ROOT.mkdir(parents=True, exist"
    "_ok=True)\n    state = read_state()\n    try:\n        if state.get('base') and health(state['base'"
    "]):\n            print(f'✅ {LABEL} 준비 완료. 아래 음성 생성 셀을 실행하세요.', flush=True)\n            return\n   "
    " except Exception:\n        pass\n    if shutil.which('nvidia-smi') is None:\n        raise Runtime"
    "Error('GPU가 없습니다. 런타임 → 런타임 유형 변경 → T4 GPU를 선택해주세요.')\n    run(['nvidia-smi', '--query-gpu=name,m"
    "emory.total', '--format=csv,noheader'], 'GPU 확인')\n    run(['apt-get', 'update', '-qq'], '시스템 패키지"
    " 목록 갱신')\n    run(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'sox', 'libsox-dev',\n         'li"
    "bsndfile1', 'build-essential', 'git'], '오디오 도구 설치')\n    if not PYTHON.is_file():\n        run([sy"
    "s.executable, '-m', 'pip', 'install', '--disable-pip-version-check', 'uv'], 'Python 환경 도구 설치')\n "
    "       run([sys.executable, '-m', 'uv', 'python', 'install', '3.10'], '별도 Python 3.10 설치')\n     "
    "   run([sys.executable, '-m', 'uv', 'venv', '--python', '3.10', '--seed', ROOT / 'venv'], '독립 실행"
    " 환경 생성')\n    if not (SOURCE / '.git').is_dir():\n        run(['git', 'clone', '--filter=blob:none"
    "', 'https://github.com/FunAudioLLM/CosyVoice.git', SOURCE], '공식 CosyVoice 소스 다운로드')\n    run(['gi"
    "t', '-C', SOURCE, 'checkout', '--detach', SOURCE_REVISION], '확인한 소스 버전 선택')\n    run(['git', '-C'"
    ", SOURCE, 'submodule', 'update', '--init', '--recursive'], 'Matcha-TTS 소스 준비')\n    # Keep upstre"
    'am versions except the documented inference-only changes below.\n    requirements = []\n    for li'
    "ne in (SOURCE / 'requirements.txt').read_text().splitlines():\n        if line.startswith(('deeps"
    "peed', 'tensorrt', 'gradio', 'fastapi-cli')):\n            continue  # optional acceleration/trai"
    "ning/UI; this server uses none of these\n        if line.startswith('diffusers=='):\n            l"
    "ine = 'diffusers==0.32.2'  # compatible with modern huggingface_hub (no cached_download import)\n"
    "        requirements.append(line)\n    requirements += ['huggingface-hub==0.30.2', 'python-multip"
    "art>=0.0.18,<0.1', 'setuptools<81']\n    req_file = ROOT / 'inference-requirements.txt'\n    req_f"
    "ile.write_text('\\n'.join(requirements) + '\\n')\n    # Whisper's pinned source distribution import"
    's pkg_resources while building.\n    # The runtime setuptools pin alone does not apply inside pip'
    "'s isolated build.\n    build_constraints = ROOT / 'build-constraints.txt'\n    build_constraints."
    "write_text('setuptools<81\\n')\n    stamp = hashlib.sha256(req_file.read_bytes() + build_constrain"
    "ts.read_bytes()).hexdigest()\n    marker = ROOT / 'dependencies.ok'\n    if not marker.exists() or"
    " marker.read_text() != stamp:\n        run([PYTHON, '-m', 'pip', 'install', 'pip>=25.3,<26', 'set"
    "uptools<81', 'wheel', 'Cython<4'], '설치 도구 준비')\n        # Install the matching CUDA pair first; d"
    "o not touch Colab's own Torch.\n        run([PYTHON, '-m', 'pip', 'install', 'torch==2.3.1', 'tor"
    "chaudio==2.3.1',\n             '--index-url', 'https://download.pytorch.org/whl/cu121'], 'GPU용 To"
    "rch 및 오디오 패키지 설치')\n        run([PYTHON, '-m', 'pip', 'install', '-r', req_file,\n             '--"
    "build-constraint', build_constraints], 'CosyVoice 의존성 설치')\n        run([PYTHON, '-m', 'pip', 'ch"
    "eck'], '의존성 충돌 확인')\n        marker.write_text(stamp)\n    download = (\n        'from huggingface_"
    'hub import snapshot_download; \'\n        f\'snapshot_download("FunAudioLLM/CosyVoice2-0.5B", revis'
    'ion={MODEL_REVISION!r}, \'\n        f\'local_dir={str(MODEL)!r}, allow_patterns=["cosyvoice2.yaml",'
    ' "llm.pt", "flow.pt", \'\n        \'"hift.pt", "campplus.onnx", "speech_tokenizer_v2.onnx", "CosyVo'
    'ice-BlankEN/*"])\'\n    )\n    run([PYTHON, \'-c\', download], \'CosyVoice 2 모델 준비 (첫 실행 시 다운로드)\')\n   '
    " missing = [name for name in ('cosyvoice2.yaml', 'llm.pt', 'flow.pt', 'hift.pt',\n               "
    "                 'campplus.onnx', 'speech_tokenizer_v2.onnx') if not (MODEL / name).is_file()]\n "
    "   if missing:\n        raise RuntimeError('모델 다운로드가 완전하지 않습니다: ' + ', '.join(missing))\n    env ="
    ' os.environ.copy()\n    # The isolated worker renders no notebook plots and may not have matplotl'
    "ib_inline installed.\n    env['MPLBACKEND'] = 'Agg'\n    env['PYTHONPATH'] = os.pathsep.join([str("
    "SOURCE), str(SOURCE / 'third_party/Matcha-TTS')])\n    libs = [str(p) for p in (ROOT / 'venv/lib/"
    "python3.10/site-packages/nvidia').glob('*/lib')]\n    env['LD_LIBRARY_PATH'] = os.pathsep.join(li"
    "bs + [env.get('LD_LIBRARY_PATH', '')])\n    env['TOKENIZERS_PARALLELISM'] = 'false'\n    env['COSY"
    "_ACCESS_TOKEN'] = secrets.token_urlsafe(24)\n    with socket.socket() as sock:\n        sock.bind("
    "('127.0.0.1', 0))\n        port = sock.getsockname()[1]\n    base = f'http://127.0.0.1:{port}/v1/{"
    'env["COSY_ACCESS_TOKEN"]}\'\n    log_path = ROOT / \'server.log\'\n    with log_path.open(\'w\') as log'
    ":\n        worker = subprocess.Popen([str(PYTHON), '-u', str(Path(__file__).resolve()), '--serve'"
    ", '--engine', ENGINE, '--port', str(port)],\n                                  cwd=SOURCE, env=en"
    "v, stdout=log, stderr=subprocess.STDOUT)\n    print('\\n▶ 모델을 GPU에 올리는 중입니다. 준비가 끝나면 완료 메시지가 나옵니다."
    "', flush=True)\n    ready = False\n    try:\n        for attempt in range(300):\n            if work"
    "er.poll() is not None:\n                raise RuntimeError('모델 시작 실패:\\n' + log_path.read_text(err"
    "ors='replace')[-7000:])\n            try:\n                ready = health(base)\n            except"
    " Exception:\n                pass\n            if ready:\n                state = {'base': base, 'p"
    "id': worker.pid, 'port': port, 'model': LABEL, 'engine': ENGINE}\n                STATE.write_tex"
    "t(json.dumps(state))\n                STATE.chmod(0o600)\n                print(f'✅ {LABEL} 준비 완료."
    " 아래 음성 생성 셀을 실행하세요.', flush=True)\n                return\n            if attempt and attempt % 15"
    " == 0:\n                print('  모델 로딩 중… 오류가 있으면 여기 표시됩니다.', flush=True)\n            time.sleep("
    "2)\n        raise RuntimeError('모델 준비 시간 초과:\\n' + log_path.read_text(errors='replace')[-7000:])\n "
    '   finally:\n        if not ready and worker.poll() is None:\n            worker.terminate()\n\n\ndef'
    ' serve(port):\n    import numpy as np\n    import soundfile as sf\n    import torch\n    from fastap'
    'i import FastAPI, File, Form, HTTPException, UploadFile\n    from fastapi.responses import Respon'
    'se\n    import uvicorn\n    import fcntl\n    if not torch.cuda.is_available():\n        raise Runti'
    "meError('CUDA GPU를 사용할 수 없습니다. T4 GPU 런타임인지 확인해주세요.')\n    from cosyvoice.cli.cosyvoice import Co"
    'syVoice2\n    model = CosyVoice2(model_dir=str(MODEL), load_jit=False, load_trt=False, fp16=False'
    ")\n    sample_rate = model.sample_rate\n    access_token = os.environ['COSY_ACCESS_TOKEN']\n    mod"
    'el_lock = threading.Lock()\n    app = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)\n\n '
    '   def authorize(token):\n        if not secrets.compare_digest(token, access_token):\n           '
    " raise HTTPException(403, '연결 주소가 올바르지 않습니다. 새 주소 전체를 복사해주세요.')\n\n    @app.get('/')\n    def index"
    "():\n        return {'service': SERVICE, 'message': LABEL + ' 실행 중. 코랩에 표시된 전체 연결 주소를 사용하세요.'}\n\n "
    "   @app.get('/v1/{token}')\n    @app.get('/v1/{token}/health')\n    def status(token: str):\n      "
    "  authorize(token)\n        return {'service': SERVICE, 'api_version': 1, 'ready': True, 'model':"
    " LABEL, 'engine': ENGINE, 'cuda': True}\n\n    @app.post('/v1/{token}/synthesize')\n    def synthes"
    "ize(token: str, text: str = Form(...), prompt_text: str = Form(''),\n                   speed: fl"
    'oat = Form(1.0), reference: UploadFile = File(...)):\n        authorize(token)\n        if not tex'
    "t.strip():\n            raise HTTPException(422, '생성할 대사를 입력해주세요.')\n        if not prompt_text.st"
    "rip():\n            raise HTTPException(422, 'CosyVoice는 참조 음성의 실제 대사도 입력해야 합니다.')\n        if len"
    "(text) > 2000:\n            raise HTTPException(422, '한 번에 2,000자 이하로 나눠 생성해주세요.')\n        if not"
    " np.isfinite(speed) or not 0.5 <= speed <= 2:\n            raise HTTPException(422, '속도는 0.5~2.0 "
    "사이여야 합니다.')\n        if not model_lock.acquire(blocking=False):\n            raise HTTPException(4"
    "09, '다른 음성을 생성 중입니다. 완료 후 다시 실행해주세요.')\n        gpu_lock = None\n        try:\n            gpu_lock"
    " = (ROOT.parent / 'gpu.lock').open('a+')\n            try:\n                fcntl.flock(gpu_lock.f"
    'ileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)\n            except BlockingIOError:\n                rais'
    "e HTTPException(409, '다른 엔진이 음성을 생성 중입니다. 완료 후 실행해주세요.')\n            payload = reference.file.re"
    'ad(10 * 1024 * 1024 + 1)\n            if not payload or len(payload) > 10 * 1024 * 1024:\n        '
    "        raise HTTPException(422, '참조 음성은 10MB 이하 WAV 또는 MP3를 사용해주세요.')\n            with tempfile"
    ".TemporaryDirectory(prefix='cosy_ref_') as folder:\n                original = Path(folder) / 're"
    "ference.audio'\n                original.write_bytes(payload)\n                prepared = Path(fol"
    "der) / 'reference.wav'\n                result = subprocess.run(['ffmpeg', '-nostdin', '-y', '-v'"
    ", 'error', '-i', str(original),\n                                         '-t', '31', '-ac', '1',"
    " '-ar', '24000', str(prepared)],\n                                        capture_output=True, te"
    'xt=True, timeout=60)\n                if result.returncode:\n                    raise HTTPExcepti'
    "on(422, '참조 오디오를 읽을 수 없습니다. WAV 또는 MP3를 확인해주세요.')\n                audio, sr = sf.read(prepared, "
    "dtype='float32')\n                duration = len(audio) / sr\n                if not 3 <= duration"
    " <= 30:\n                    raise HTTPException(422, f'참조 음성은 3~30초여야 합니다. 현재 {duration:.1f}초입니다"
    ".')\n                if not np.all(np.isfinite(audio)) or np.max(np.abs(audio)) < 0.00001:\n      "
    "              raise HTTPException(422, '참조 음성에 들리는 목소리가 없습니다.')\n                pieces = []\n    "
    "            sentences = re.split(r'(?<=[.!?。！？])\\s+|\\n+', text.strip())\n                chunks ="
    ' []\n                for sentence in sentences:\n                    sentence = sentence.strip()\n '
    "                   while len(sentence) > 180:\n                        cut = sentence.rfind(' ', "
    '60, 180)\n                        cut = cut if cut >= 60 else 180\n                        chunks.'
    'append(sentence[:cut])\n                        sentence = sentence[cut:].strip()\n               '
    '     if sentence:\n                        chunks.append(sentence)\n                with torch.inf'
    'erence_mode():\n                    for chunk in chunks:\n                        for item in mode'
    'l.inference_zero_shot(chunk, prompt_text.strip(), str(prepared),\n                               '
    '                              stream=False, speed=speed, text_frontend=False):\n                 '
    "           pieces.append(item['tts_speech'].detach().cpu().numpy().reshape(-1))\n                "
    "if not pieces or sum(x.size for x in pieces) == 0:\n                    raise RuntimeError('모델이 빈"
    " 음성을 반환했습니다. 참조 음성과 실제 대사를 확인해주세요.')\n                speech = np.concatenate(pieces)\n           "
    '     if not np.all(np.isfinite(speech)) or np.max(np.abs(speech)) < 0.000001:\n                  '
    "  raise RuntimeError('모델 출력이 무음이거나 손상되었습니다. server.log를 확인해주세요.')\n                output = io.By"
    "tesIO()\n                sf.write(output, speech, sample_rate, format='WAV', subtype='PCM_16')\n  "
    "              return Response(output.getvalue(), media_type='audio/wav')\n        except HTTPExce"
    "ption:\n            raise\n        except Exception as exc:\n            logging.exception('%s synt"
    "hesis failed', LABEL)\n            raise HTTPException(500, f'{type(exc).__name__}: {str(exc)[:60"
    "0]}') from exc\n        finally:\n            reference.file.close()\n            if gpu_lock is no"
    't None:\n                gpu_lock.close()\n            model_lock.release()\n\n    uvicorn.run(app, '
    "host='127.0.0.1', port=port, access_log=False)\n\n\ndef tunnel():\n    state = read_state()\n    if n"
    "ot state.get('base') or not health(state['base']):\n        raise RuntimeError('먼저 1번 설치 셀을 실행해주세"
    "요.')\n    if state.get('public_base'):\n        try:\n            if health(state['public_base']):\n"
    "                print(LABEL + ' 프로그램 연결 주소:\\n' + state['public_base'], flush=True)\n             "
    "   return\n        except Exception:\n            pass\n    executable = ROOT / 'cloudflared'\n    i"
    "f not executable.is_file():\n        print('연결 도구 다운로드 중…', flush=True)\n        temp = executable"
    ".with_suffix('.download')\n        urllib.request.urlretrieve('https://github.com/cloudflare/clou"
    "dflared/releases/latest/download/cloudflared-linux-amd64', temp)\n        temp.chmod(0o755)\n     "
    "   temp.replace(executable)\n    log_path = ROOT / 'tunnel.log'\n    with log_path.open('w') as lo"
    "g:\n        proc = subprocess.Popen([str(executable), 'tunnel', '--no-autoupdate', '--url',\n     "
    '                            f\'http://127.0.0.1:{state["port"]}\'], stdout=log, stderr=subprocess.'
    'STDOUT)\n    success = False\n    try:\n        deadline = time.monotonic() + 120\n        while tim'
    'e.monotonic() < deadline:\n            if proc.poll() is not None:\n                raise RuntimeE'
    "rror('연결 도구가 종료되었습니다:\\n' + log_path.read_text(errors='replace')[-2000:])\n            match = re."
    "search(r'https://[a-z0-9-]+\\.trycloudflare\\.com', log_path.read_text(errors='replace'))\n        "
    "    if match:\n                public = match.group(0) + state['base'].split(str(state['port']), "
    '1)[1]\n                try:\n                    if health(public):\n                        state.'
    'update(public_base=public, tunnel_pid=proc.pid)\n                        STATE.write_text(json.du'
    "mps(state))\n                        print('\\n✅ ' + LABEL + ' 프로그램 연결 준비 완료\\n프로그램 연결 주소:\\n' + pub"
    "lic, flush=True)\n                        print(f'위 주소 전체를 AI Voice Studio의 {LABEL} 접속 주소에 붙여 넣으세"
    "요.', flush=True)\n                        success = True\n                        return\n         "
    '       except Exception:\n                    pass\n            time.sleep(2)\n        raise Runtim'
    "eError('외부 연결을 열지 못했습니다. 코랩 내부의 2~3번 셀은 계속 사용할 수 있습니다.')\n    finally:\n        if not success and"
    ' proc.poll() is None:\n            proc.terminate()\n\n\ndef main():\n    parser = argparse.ArgumentP'
    "arser()\n    modes = parser.add_mutually_exclusive_group(required=True)\n    modes.add_argument('-"
    "-setup', action='store_true')\n    modes.add_argument('--serve', action='store_true')\n    modes.a"
    "dd_argument('--tunnel', action='store_true')\n    parser.add_argument('--port', type=int, default"
    "=50000)\n    parser.add_argument('--engine', choices=['cosyvoice'], default='cosyvoice')\n    args"
    ' = parser.parse_args()\n    if args.setup:\n        try:\n            setup()\n        except Except'
    "ion:\n            if ROOT.is_dir():\n                with (ROOT / 'setup.log').open('a', encoding="
    "'utf-8') as log:\n                    traceback.print_exc(file=log)\n            raise\n    elif ar"
    "gs.serve:\n        serve(args.port)\n    else:\n        tunnel()\n\n\nif __name__ == '__main__':\n    m"
    'ain()\n'
)
server_script = Path('/content/ai_voice_cosy_runner.py')
server_script.write_text(server_source, encoding='utf-8')
log_dir = Path('/content/ai_voice_dual_v1/cosyvoice')
log_dir.mkdir(parents=True, exist_ok=True)
recent = deque(maxlen=55)
with (log_dir / 'notebook.log').open('w', encoding='utf-8') as log:
    process = subprocess.Popen([sys.executable, '-u', str(server_script), '--setup'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
            recent.append(line)
        code = process.wait()
    except BaseException:
        process.terminate()
        process.wait()
        raise
if code:
    print('\n【설치가 멈춘 실제 오류 · 이 부분을 보내주세요】\n' + ''.join(recent), flush=True)
    raise RuntimeError('설치가 중단되었습니다. 위 【실제 오류】 내용을 확인해주세요.') from None
print('✅ CosyVoice 2 준비 완료! 사이트 연결은 4번, 코랩에서 직접 만들기는 2·3번을 실행하세요.')


In [ ]:
#@title 2. 참고 음성 업로드 — 목소리를 바꿀 때 실행
from google.colab import files
from pathlib import Path
import hashlib
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('참고 음성 WAV 또는 MP3 파일 한 개를 선택해주세요.')
name, contents = next(iter(uploaded.items()))
if Path(name).suffix.lower() not in ('.wav', '.mp3'):
    raise ValueError('WAV 또는 MP3 파일을 선택해주세요.')
if len(contents) > 10 * 1024 * 1024:
    raise ValueError('3~30초, 10MB 이하의 음성을 선택해주세요.')
reference_dir = Path('/content/ai_voice_dual_v1/references')
reference_dir.mkdir(parents=True, exist_ok=True)
reference_path = reference_dir / (hashlib.sha256(contents).hexdigest()[:16] + Path(name).suffix.lower())
reference_path.write_bytes(contents)
print('참고 음성 준비 완료:', name)
print('이제 3번에서 참고 음성의 실제 대사와 새로 읽힐 대사를 입력하세요.')


In [ ]:
#@title 3. 대사 입력 → CosyVoice 음성 생성·다운로드
새로_읽힐_대사 = "옛날 어느 산골 마을에 마음씨 착한 청년이 살고 있었습니다." #@param {type:"string"}
참고음성_실제대사 = "" #@param {type:"string"}
말하기_속도 = 1.0 #@param {type:"number"}
import json
import io
import uuid
import wave
from pathlib import Path
import requests
from IPython.display import Audio, display
from google.colab import files
engine = 'cosyvoice'
prompt = 참고음성_실제대사.strip()
if not prompt:
    raise ValueError('참고 음성에서 실제로 말한 내용을 그대로 입력해주세요.')
if not 새로_읽힐_대사.strip():
    raise ValueError('새로 읽힐 대사를 입력해주세요.')
if 'reference_path' not in globals() or not reference_path.is_file():
    raise ValueError('먼저 2번에서 참고 음성을 업로드해주세요.')
state_file = Path('/content/ai_voice_dual_v1') / engine / 'state.json'
if not state_file.is_file():
    raise RuntimeError('CosyVoice가 준비되지 않았습니다. 먼저 1번을 실행해주세요.')
state = json.loads(state_file.read_text())
print('CosyVoice 음성 생성 중…')
try:
    with reference_path.open('rb') as audio:
        response = requests.post(state['base'] + '/synthesize',
            data={'text': 새로_읽힐_대사, 'prompt_text': prompt, 'speed': 말하기_속도},
            files={'reference': (reference_path.name, audio, 'application/octet-stream')},
            timeout=(10, 600))
except requests.RequestException as exc:
    raise RuntimeError('코랩 연결이 끊겼거나 응답 시간이 초과되었습니다. 1번 준비 상태와 마지막 로그를 확인해주세요.') from exc
if not response.ok:
    try:
        message = response.json().get('detail', response.text[:400])
    except ValueError:
        message = response.text[:400]
    raise RuntimeError(f'음성 생성 실패 ({response.status_code}): {message}')
with wave.open(io.BytesIO(response.content), 'rb') as wav:
    if wav.getnframes() == 0:
        raise RuntimeError('빈 음성이 반환되었습니다. 아래 오류 확인 셀을 실행해주세요.')
output_dir = Path('/content/ai_voice_dual_v1/outputs')
output_dir.mkdir(exist_ok=True)
output = output_dir / (engine + '_' + uuid.uuid4().hex[:8] + '.wav')
output.write_bytes(response.content)
display(Audio(filename=str(output)))
print('생성 완료. 음성을 듣고 다운로드하세요.')
files.download(str(output))


## 선택: 공유 사이트의 AI Voice Studio에 연결

아래 **프로그램에_연결**을 체크하고 왼쪽 ▶를 누르세요.
나온 **프로그램 연결 주소 전체**를 사이트 왼쪽 **CosyVoice 2 · 내 코랩 주소** 칸에 붙여 넣으세요.
주소 뒤의 `/v1/…` 부분도 포함합니다.

주소에는 임시 접속 키가 포함됩니다. 본인만 사용하고, 런타임 종료 후에는 새 주소를 받으세요.
무료 코랩에서는 외부 웹 화면을 통한 사용이 제한되어 세션이 종료될 수 있습니다.
연결이 안 되면 코랩 2·3번에서 직접 생성할 수 있습니다. [Colab 안내](https://research.google.com/colaboratory/faq.html)


In [ ]:
#@title 4. 선택: 사이트에 붙여 넣을 연결 주소 받기
프로그램에_연결 = False #@param {type:"boolean"}
if 프로그램에_연결:
    from pathlib import Path
    import subprocess
    import sys
    script = Path('/content/ai_voice_cosy_runner.py')
    if not script.is_file():
        raise RuntimeError('먼저 1번 설치 셀을 실행해주세요.')
    result = subprocess.run([sys.executable, '-u', str(script), '--tunnel'])
    if result.returncode:
        raise RuntimeError('연결하지 못했습니다. 바로 위 오류 또는 아래 오류 확인 셀을 확인해주세요.')
else:
    print('사이트에 연결하려면 체크 후 실행하세요. 코랩에서 직접 만들려면 2·3번을 사용하세요.')


In [ ]:
#@title 오류가 날 때만 실행 · 출력 내용을 보내주세요
from pathlib import Path
for name in ('notebook.log', 'setup.log', 'server.log'):
    log = Path('/content/ai_voice_dual_v1/cosyvoice') / name
    print('\n【' + name + '】')
    print('\n'.join(log.read_text(errors='replace').splitlines()[-45:]) if log.is_file() else '아직 기록 없음')


### 실행 방식

CosyVoice는 별도의 Python 3.10 환경에서 실행합니다. 설치 오류가 나면 중지하며 모델 로딩이 끝난 뒤에 준비 완료를 표시합니다.
음성 생성이 실패해도 같은 요청을 자동으로 반복하지 않습니다.

코드 문법과 앱 연결 경로는 확인했으며, 실제 Colab GPU 음성 생성은 별도 확인이 필요합니다.
공식 자료: [CosyVoice](https://github.com/FunAudioLLM/CosyVoice).
